# 🚀 JobLens — Complete Unified Notebook

This notebook merges:
- **Part A**: Job Scraper (Wuzzuf + LinkedIn → ChromaDB)
- **Part B**: CV Parser + ATS Scorer + Candidate API
- **Shared ChromaDB**: one database at `./joblens_db` with two collections:
  - `job_listings` — scraped jobs
  - `candidates` — parsed candidate profiles
- **One unified FastAPI** server exposing all endpoints

### Execution Order
1. Cell 1 — Install packages
2. Cell 2 — Imports
3. Cell 3 — Config, API keys, embedding models, AI Detector
4. Cell 4 — Shared ChromaDB init
5. Cell 5 — Scraper helpers & `JobLensScraper` class
6. Cell 6 — CV parsing helpers (PyMuPDF, Docling, LLM)
7. Cell 7 — ATS analysis + Improvements
8. Cell 8 — `JobMatcher` (now queries real DB)
9. Cell 9 — Run scraper (optional, skip if DB already populated)
10. Cell 10 — Run CV analysis interactively
11. Cell 11 — Unified FastAPI server
12. Cell 12 — Expose via ngrok

In [ ]:
# ============================================================
# CELL 1 — INSTALL ALL DEPENDENCIES
# ============================================================

# Core scraping
!pip install -q playwright playwright-stealth nest_asyncio
!playwright install chromium
!playwright install --with-deps

# Core ML & CV parsing
!pip install -q openai python-docx pillow sentence-transformers scikit-learn
!pip install -q PyMuPDF
!pip install -q docling
!pip install -q textstat
!pip install -q transformers torch

# API & DB
!pip install -q fastapi uvicorn pydantic nest-asyncio python-multipart
!pip install -q chromadb
!pip install -q apscheduler
!pip install -q pyngrok

print('✅ All packages installed.')

Installing dependencies...
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 4,271 B in 3s (1,279 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building depende

In [ ]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import sys
import os
import asyncio
import json
import base64
import hashlib
import re
import random
import urllib.parse
import tempfile
import threading
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any
from io import BytesIO

import nest_asyncio
nest_asyncio.apply()

# Windows fix for Playwright
if sys.platform.startswith('win'):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# Scraping
from playwright.async_api import async_playwright
try:
    from playwright_stealth import stealth_async
    STEALTH_AVAILABLE = True
except ImportError:
    STEALTH_AVAILABLE = False

# File processing
import fitz          # PyMuPDF
import docx
from PIL import Image

# Docling
try:
    from docling.document_converter import DocumentConverter
    DOCLING_AVAILABLE = True
except ImportError:
    print('⚠️  Docling not available, using fallback methods')
    DOCLING_AVAILABLE = False

# ML
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import textstat
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# DB
import chromadb

# API
from openai import OpenAI
from fastapi import FastAPI, UploadFile, File, HTTPException, BackgroundTasks, Query
from pydantic import BaseModel
import uvicorn
from apscheduler.schedulers.asyncio import AsyncIOScheduler
from contextlib import asynccontextmanager

# Colab helpers
try:
    from google.colab import userdata, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('⚠️  Not running in Colab — userdata and files.upload() unavailable.')

print('✅ All imports successful.')

✅ All imports successful.


In [ ]:
# ============================================================
# CELL 3 — CONFIGURATION, API KEYS, MODELS, AI DETECTOR
# ============================================================

# ── Paths & constants ──────────────────────────────────────
CHROMA_PATH     = './joblens_db'          # single shared DB for the whole project
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'     # used by the scraper
CONCURRENCY_LIMIT = 5

TARGET_CATEGORIES = [
    'Data Science', 'Engineering', 'Information Technology', 'Software Development',
    'Web Development', 'Mobile Development', 'Data Engineering', 'Data Analytics',
    'Artificial Intelligence', 'Machine Learning Engineering', 'Cloud Computing',
    'DevOps Engineering', 'Site Reliability Engineering (SRE)', 'Cybersecurity',
    'Information Security', 'Network Engineering', 'System Administration',
    'Database Administration', 'Big Data Engineering', 'Blockchain Development',
    'Embedded Systems Engineering', 'Game Development', 'IT Support / Technical Support',
    'IT Infrastructure', 'Automation Engineering', 'Quality Engineering (Software Testing)',
    'UI/UX Design', 'Product Engineering', 'Robotics Engineering', 'AR/VR Development',
    'Platform Engineering', 'API Development', 'Technical Architecture',
    'Product Management', 'Finance', 'Marketing', 'Sales', 'Business Development'
]

# ── LLM provider ───────────────────────────────────────────
PROVIDER = 'openrouter'   # 'openrouter' or 'groq'

try:
    GROQ_API_KEY        = userdata.get('GROQ_API_KEY')
    OPENROUTER_API_KEY  = userdata.get('OPENROUTER_API_KEY')
    print('🔐 API Keys retrieved from Colab Secrets.')
except Exception:
    GROQ_API_KEY        = os.getenv('GROQ_API_KEY', '')
    OPENROUTER_API_KEY  = os.getenv('OPENROUTER_API_KEY', '')
    print('⚠️  Using environment variables for API keys.')

if PROVIDER == 'openrouter':
    if not OPENROUTER_API_KEY:
        raise ValueError('🚨 OPENROUTER_API_KEY is missing!')
    llm_client = OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url='https://openrouter.ai/api/v1',
    )
    PARSING_MODEL  = 'meta-llama/llama-3.3-70b-instruct'
    PARSING_FALLBACK = 'meta-llama/llama-3.1-70b-instruct'
    ATS_MODEL      = 'meta-llama/llama-3.3-70b-instruct'
    SCORING_MODEL  = 'meta-llama/llama-3.3-70b-instruct'
    OCR_MODEL      = 'meta-llama/llama-3.2-90b-vision-preview'
    print('🎯 Using OpenRouter backend')
else:
    if not GROQ_API_KEY:
        raise ValueError('🚨 GROQ_API_KEY is missing!')
    llm_client = OpenAI(
        api_key=GROQ_API_KEY,
        base_url='https://api.groq.com/openai/v1'
    )
    PARSING_MODEL  = 'llama-3.3-70b-versatile'
    PARSING_FALLBACK = 'llama-3.1-70b-versatile'
    ATS_MODEL      = 'llama-3.3-70b-versatile'
    SCORING_MODEL  = 'llama-3.3-70b-versatile'
    OCR_MODEL      = 'llama-3.2-90b-vision-preview'
    print('⚡ Using Groq backend')

# ── Embedding model (used by CV matcher) ──────────────────
print('🔄 Loading CV embedding model (all-mpnet-base-v2)...')
cv_embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
print('✅ CV embedding model loaded.')

# ── AI Content Detector ────────────────────────────────────
class AIDetector:
    """Perplexity + Burstiness AI-content detector using GPT-2."""

    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f'🕵️  Initialising AI Detector on {self.device}...')
        model_id = 'gpt2'
        self.model     = GPT2LMHeadModel.from_pretrained(model_id).to(self.device)
        self.tokenizer = GPT2TokenizerFast.from_pretrained(model_id)

    def calculate_perplexity(self, text: str) -> float:
        enc = self.tokenizer(text, return_tensors='pt')
        ids = enc.input_ids.to(self.device)
        with torch.no_grad():
            out = self.model(ids, labels=ids)
        return torch.exp(out.loss).item()

    def analyze_text(self, text: str) -> Dict:
        text = re.sub(r'\n+', ' ', text).strip()
        if not text:
            return {'ai_probability_score': 0, 'verdict': 'Empty text',
                    'avg_perplexity': 0, 'burstiness_score': 0, 'sentence_count': 0}

        sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 10]
        if len(sentences) < 3:
            ppl = self.calculate_perplexity(text)
            return {'ai_probability_score': 50, 'verdict': 'Insufficient Data',
                    'avg_perplexity': round(ppl, 2), 'burstiness_score': 0, 'sentence_count': len(sentences)}

        perplexities = []
        for s in sentences[:20]:
            try:
                perplexities.append(self.calculate_perplexity(s))
            except Exception:
                continue

        if not perplexities:
            return {'ai_probability_score': 0, 'verdict': 'Error',
                    'avg_perplexity': 0, 'burstiness_score': 0, 'sentence_count': 0}

        avg_ppl    = np.mean(perplexities)
        burstiness = np.std(perplexities)

        score = 0
        if avg_ppl  < 40:  score += 60
        elif avg_ppl < 70:  score += 40
        elif avg_ppl < 100: score += 20
        if burstiness < 15: score += 40
        elif burstiness < 30: score += 20
        score = min(score, 99)

        if score > 80:   verdict = '🤖 Likely AI Generated'
        elif score > 50: verdict = '❓ Mixed / Suspicious'
        else:            verdict = '👤 Likely Human Written'

        return {
            'ai_probability_score': score,
            'verdict':              verdict,
            'avg_perplexity':       round(avg_ppl, 2),
            'burstiness_score':     round(burstiness, 2),
            'sentence_count':       len(sentences)
        }

ai_detector = AIDetector()
print('✅ Configuration complete.')

🔐 API Keys retrieved from Colab Secrets.
🎯 Using OpenRouter backend
🔄 Loading CV embedding model (all-mpnet-base-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ CV embedding model loaded.
🕵️  Initialising AI Detector on cpu...
✅ Configuration complete.


In [ ]:
# ============================================================
# CELL 4 — SHARED CHROMADB INITIALISATION
# ============================================================
#
# One persistent client, two collections:
#   • job_listings  — populated by the scraper
#   • candidates    — populated by the CV parser / API

print(f'🗄️  Opening ChromaDB at {CHROMA_PATH} …')
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

jobs_col       = chroma_client.get_or_create_collection(name='job_listings')
candidates_col = chroma_client.get_or_create_collection(name='candidates')

print(f'✅ job_listings  collection: {jobs_col.count()} documents')
print(f'✅ candidates    collection: {candidates_col.count()} documents')

🗄️  Opening ChromaDB at ./joblens_db …
✅ job_listings  collection: 1734 documents
✅ candidates    collection: 0 documents


In [ ]:
# ============================================================
# CELL 5 — SCRAPER HELPERS + JobLensScraper CLASS
# ============================================================

# ── Scraper embedding model (lightweight) ─────────────────
print('🔄 Loading scraper embedding model (all-MiniLM-L6-v2)...')
scraper_embed_model = SentenceTransformer(EMBEDDING_MODEL)
print('✅ Scraper embedding model loaded.')


class JobLensScraper:
    """Scrapes Wuzzuf and LinkedIn, stores results in jobs_col."""

    def __init__(self):
        self.collection  = jobs_col          # shared collection
        self.model       = scraper_embed_model
        existing         = self.collection.get()
        self.existing_ids = set(existing['ids'])
        print(f'📋 Loaded {len(self.existing_ids)} existing jobs from DB.')

    def normalize_text(self, text: str) -> str:
        return re.sub(r'[^a-z0-9]', '', text.lower()) if text else ''

    def generate_job_id(self, title: str, company: str) -> str:
        raw = f'{self.normalize_text(title)}|{self.normalize_text(company)}'
        return hashlib.md5(raw.encode()).hexdigest()

    def save_jobs(self, jobs: List[Dict]):
        if not jobs:
            return
        batch_dedup: Dict[str, Dict] = {}
        print(f'   Processing {len(jobs)} candidates for database...')

        for job in jobs:
            job_id = self.generate_job_id(job['title'], job['company'])

            if job_id in self.existing_ids:
                rec = self.collection.get(ids=[job_id])
                if rec and rec['metadatas']:
                    old_meta = rec['metadatas'][0]
                    if job.get('posted_time', '') > old_meta.get('posted_time', ''):
                        print(f"   🔄 Repost detected: updating '{job['title']}'")
                    else:
                        continue
            else:
                self.existing_ids.add(job_id)

            skills_str = ', '.join(job.get('skills', []))
            text_for_embed = (
                f"{job['title']} at {job['company']}. "
                f"Location: {job['location']}. "
                f"Skills: {skills_str}. "
                f"{job.get('description', '')}"
            )

            meta = {
                'source':           job['source'],
                'title':            job['title'],
                'company':          job['company'],
                'location':         job['location'],
                'job_page_link':    job['job_page_link'],
                'apply_link':       job.get('apply_link', job['job_page_link']),
                'posted_time':      job.get('posted_time', str(datetime.now().date())),
                'experience_level': job.get('experience_level', 'Not Specified'),
                'employment_type':  job.get('employment_type', 'Not Specified'),
                'skills_list':      skills_str,
                'description_snippet': job.get('description', '')[:200],
                'json_detailed':    json.dumps(job)
            }

            batch_dedup[job_id] = {
                'doc':       text_for_embed,
                'meta':      meta,
                'embedding': self.model.encode(text_for_embed).tolist()
            }

        if batch_dedup:
            self.collection.upsert(
                ids=list(batch_dedup.keys()),
                embeddings=[v['embedding'] for v in batch_dedup.values()],
                metadatas=[v['meta']       for v in batch_dedup.values()],
                documents=[v['doc']        for v in batch_dedup.values()]
            )
            print(f'   ✅ Upserted {len(batch_dedup)} jobs to DB.')
        else:
            print('   ⚠️  All jobs were standard duplicates. No updates needed.')


# ── Date / description helpers ─────────────────────────────

def parse_relative_date(text: str) -> str:
    if not text:
        return str(datetime.now().date())
    text  = text.lower()
    today = datetime.now()
    try:
        if 'hour' in text or 'minute' in text or 'just now' in text:
            delta = timedelta(days=0)
        elif 'day' in text:
            delta = timedelta(days=int(re.search(r'\d+', text).group()))
        elif 'week' in text:
            delta = timedelta(weeks=int(re.search(r'\d+', text).group()))
        elif 'month' in text:
            delta = timedelta(days=int(re.search(r'\d+', text).group()) * 30)
        else:
            delta = timedelta(days=0)
        return (today - delta).strftime('%Y-%m-%d')
    except Exception:
        return str(today.date())


def parse_description(full_text: str):
    lower = full_text.lower()
    req_start  = lower.find('requirements')  if 'requirements'  in lower else lower.find('qualifications')
    resp_start = lower.find('responsibilities') if 'responsibilities' in lower else lower.find('duties')
    requirements, responsibilities = '', ''
    if req_start != -1 and resp_start != -1:
        if req_start < resp_start:
            requirements, responsibilities = full_text[req_start:resp_start], full_text[resp_start:]
        else:
            responsibilities, requirements = full_text[resp_start:req_start], full_text[req_start:]
    elif req_start  != -1: requirements    = full_text[req_start:]
    elif resp_start != -1: responsibilities = full_text[resp_start:]
    return requirements.strip(), responsibilities.strip()


async def block_resources(route):
    if route.request.resource_type in ['image', 'media', 'font', 'stylesheet']:
        await route.abort()
    else:
        await route.continue_()


async def get_job_details(context, link: str, source: str) -> Dict:
    if not link:
        return {}
    await asyncio.sleep(random.uniform(0.5, 1.5))
    page = await context.new_page()
    await page.route('**/*', block_resources)

    data = {
        'description': '', 'requirements': '', 'responsibilities': '',
        'skills': [], 'experience_level': 'Not Specified',
        'employment_type': 'Not Specified', 'apply_link': link
    }

    try:
        await page.goto(link, timeout=15000)

        if source == 'Wuzzuf':
            try:
                elm = await page.wait_for_selector('.css-1uobp1k', timeout=3000)
                data['description'] = await elm.inner_text()
            except Exception:
                data['description'] = 'Description not found.'
            try:
                tags = await page.query_selector_all('.css-158idm4, a.css-o171kl')
                data['skills'] = [(await t.inner_text()).strip() for t in tags]
            except Exception:
                pass
            try:
                ce = await page.query_selector("span:has-text('Career Level:') + span")
                if ce: data['experience_level'] = await ce.inner_text()
                te = await page.query_selector("span:has-text('Job Type:') + span")
                if te: data['employment_type'] = await te.inner_text()
            except Exception:
                pass

        elif source == 'LinkedIn':
            try:
                try: await page.click('button.show-more-less-html__button', timeout=1000)
                except Exception: pass
                elm = await page.wait_for_selector('.show-more-less-html__markup, .description__text', timeout=3000)
                data['description'] = await elm.inner_text()
            except Exception:
                data['description'] = 'Description not found.'
            try:
                btn = await page.query_selector('a.top-card-layout__cta--primary, a.apply-button')
                if btn:
                    href = await btn.get_attribute('href')
                    if href and 'linkedin.com' not in href:
                        data['apply_link'] = href
            except Exception:
                pass
            try:
                criteria = await page.query_selector_all('.description__job-criteria-item')
                for item in criteria:
                    h = await item.query_selector('h3')
                    v = await item.query_selector('span')
                    if h and v:
                        ht, vt = (await h.inner_text()).lower(), (await v.inner_text()).strip()
                        if 'seniority'  in ht: data['experience_level'] = vt
                        if 'employment' in ht: data['employment_type']  = vt
            except Exception:
                pass

        if data['description']:
            reqs, resps = parse_description(data['description'])
            data['requirements']    = reqs  or 'See Description'
            data['responsibilities'] = resps or 'See Description'

    except Exception:
        pass
    finally:
        await page.close()
    return data


# ── Main scraper coroutine ─────────────────────────────────

async def run_scraper():
    joblens = JobLensScraper()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)   # headless=True for Colab
        context = await browser.new_context(
            user_agent=(
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                'AppleWebKit/537.36 (KHTML, like Gecko) '
                'Chrome/121.0.0.0 Safari/537.36'
            )
        )

        for category in TARGET_CATEGORIES:
            print(f'\n🚀 Scraping category: {category}')
            enc = urllib.parse.quote(category)

            # ── Wuzzuf ──────────────────────────────────────
            try:
                page = await context.new_page()
                await page.route('**/*', block_resources)
                url  = (f'https://wuzzuf.net/search/jobs/?q={enc}'
                        f'&a=hpb&filters%5Bcountry%5D%5B0%5D=Egypt')
                await page.goto(url, timeout=60000)
                try:
                    await page.wait_for_selector('.css-1gatmva, div.css-pkv5jc, .css-bjn8wh', timeout=15000)
                except Exception:
                    print('   ⚠️  Wuzzuf: no jobs or timeout.')

                cards = await page.query_selector_all('.css-1gatmva')
                if not cards: cards = await page.query_selector_all('div.css-pkv5jc')
                if not cards: cards = await page.query_selector_all('div.css-bjn8wh')
                print(f'   [Wuzzuf] Found {len(cards)} job cards.')

                basics = []
                for card in cards:
                    try:
                        t_el = await card.query_selector('h2') or await card.query_selector('h1')
                        title = (await t_el.inner_text()).strip() if t_el else 'N/A'
                        c_el  = await card.query_selector('a.css-p7pghv') or await card.query_selector('a.css-17s97q8')
                        company = (await c_el.inner_text()).strip().replace(' -', '') if c_el else 'Confidential'
                        l_el  = await card.query_selector('strong.css-1vlp604')
                        if l_el:
                            ft = (await l_el.inner_text()).strip()
                            location = ft.replace(company, '').replace('-', '').strip()
                        else:
                            l_old = await card.query_selector('.css-5wys0k')
                            location = (await l_old.inner_text()).strip() if l_old else 'Egypt'
                        d_el = await card.query_selector('.css-do6t5g, .css-4c4ojb, .css-154erwh')
                        date_txt = await d_el.inner_text() if d_el else ''
                        link_el  = await card.query_selector('h2 a') or await card.query_selector('h1 a')
                        post_link = ''
                        if link_el: post_link = await link_el.get_attribute('href')
                        elif c_el:  post_link = await c_el.get_attribute('href')
                        basics.append({'source': 'Wuzzuf', 'title': title,
                                       'company': company, 'location': location,
                                       'posted_time': parse_relative_date(date_txt),
                                       'job_page_link': post_link})
                    except Exception:
                        continue

                await page.close()
                for i in range(0, len(basics), CONCURRENCY_LIMIT):
                    batch   = basics[i: i + CONCURRENCY_LIMIT]
                    details = await asyncio.gather(*[get_job_details(context, b['job_page_link'], 'Wuzzuf') for b in batch])
                    for idx, d in enumerate(details): batch[idx].update(d)
                    joblens.save_jobs(batch)

            except Exception as e:
                print(f'❌ Wuzzuf error on {category}: {e}')

            # ── LinkedIn ─────────────────────────────────────
            try:
                page = await context.new_page()
                await page.route('**/*', block_resources)
                url  = (f'https://www.linkedin.com/jobs/search?keywords={enc}'
                        f'&location=Egypt&position=1&pageNum=0')
                await page.goto(url, timeout=60000)

                prev_count, no_change = 0, 0
                for _ in range(8):
                    await page.mouse.wheel(0, 1500)
                    await asyncio.sleep(1.5)
                    cards = await page.query_selector_all('.base-card')
                    if len(cards) == prev_count:
                        no_change += 1
                        if no_change >= 2: break
                    else:
                        no_change = 0
                    prev_count = len(cards)

                cards = await page.query_selector_all('.base-card')
                print(f'   [LinkedIn] Found {len(cards)} job cards.')

                basics = []
                for card in cards:
                    try:
                        t_el = await card.query_selector('.base-search-card__title')
                        c_el = await card.query_selector('.base-search-card__subtitle')
                        l_el = await card.query_selector('.job-search-card__location')
                        d_el = await card.query_selector('time')
                        lnk  = await card.query_selector('a.base-card__full-link')
                        if t_el and lnk:
                            date_attr = await d_el.get_attribute('datetime') if d_el else ''
                            if not date_attr and d_el: date_attr = (await d_el.inner_text()).strip()
                            basics.append({
                                'source':        'LinkedIn',
                                'title':         (await t_el.inner_text()).strip(),
                                'company':       (await c_el.inner_text()).strip() if c_el else 'N/A',
                                'location':      (await l_el.inner_text()).strip() if l_el else 'Egypt',
                                'posted_time':   date_attr or str(datetime.now().date()),
                                'job_page_link': await lnk.get_attribute('href')
                            })
                    except Exception:
                        continue

                await page.close()
                for i in range(0, len(basics), CONCURRENCY_LIMIT):
                    batch   = basics[i: i + CONCURRENCY_LIMIT]
                    details = await asyncio.gather(*[get_job_details(context, b['job_page_link'], 'LinkedIn') for b in batch])
                    for idx, d in enumerate(details): batch[idx].update(d)
                    joblens.save_jobs(batch)

            except Exception as e:
                print(f'❌ LinkedIn error on {category}: {e}')

            sleep_t = random.uniform(25, 45)
            print(f'💤 Resting {int(sleep_t)}s before next category…')
            await asyncio.sleep(sleep_t)

        await browser.close()
    print('🏁 Scraping complete.')


print('✅ Scraper helpers defined.')

🔄 Loading scraper embedding model (all-MiniLM-L6-v2)...
✅ Scraper embedding model loaded.
✅ Scraper helpers defined.


In [ ]:
# ============================================================
# CELL 6 — CV FILE PARSING HELPERS
# ============================================================

# ── PyMuPDF ───────────────────────────────────────────────
def extract_text_with_pymupdf(file_path: str) -> Dict:
    try:
        doc = fitz.open(file_path)
        pages, full = [], []
        for i, page in enumerate(doc):
            t = page.get_text('text')
            full.append(t)
            pages.append({'page_number': i + 1, 'text': t, 'text_length': len(t)})
        result = {'text': '\n'.join(full), 'pages': pages,
                  'metadata': doc.metadata, 'page_count': len(doc)}
        doc.close()
        print(f'✅ PyMuPDF: {len(result["text"])} chars from {result["page_count"]} page(s)')
        return result
    except Exception as e:
        print(f'❌ PyMuPDF error: {e}')
        return {'text': '', 'pages': [], 'metadata': {}, 'page_count': 0}


def extract_links_from_pdf(file_path: str) -> Dict[str, list]:
    try:
        doc = fitz.open(file_path)
        linkedin, github, others = [], [], []
        for page in doc:
            for link in page.get_links():
                uri = link.get('uri', '')
                if uri:
                    if 'linkedin.com' in uri.lower()  and uri not in linkedin: linkedin.append(uri)
                    elif 'github.com' in uri.lower()  and uri not in github:   github.append(uri)
                    elif uri not in others:                                     others.append(uri)
        doc.close()
        if linkedin or github:
            print(f'🔗 Links found — LinkedIn: {len(linkedin)}, GitHub: {len(github)}')
        return {'linkedin': linkedin, 'github': github, 'others': others}
    except Exception as e:
        print(f'❌ Link extraction error: {e}')
        return {'linkedin': [], 'github': [], 'others': []}


# ── Docling ───────────────────────────────────────────────
def extract_with_docling(file_path: str) -> Optional[Dict]:
    if not DOCLING_AVAILABLE:
        return None
    try:
        print('🔄 Processing with Docling…')
        result = DocumentConverter().convert(file_path)
        d = {'text':      result.document.export_to_text(),
             'markdown':  result.document.export_to_markdown(),
             'structure': str(result.document)}
        print(f'✅ Docling: {len(d["text"])} chars')
        return d
    except Exception as e:
        print(f'⚠️  Docling error: {e}')
        return None


# ── DOCX ─────────────────────────────────────────────────
def extract_text_from_docx(file_path: str) -> str:
    try:
        doc  = docx.Document(file_path)
        paras = [p.text for p in doc.paragraphs if p.text.strip()]
        tables = []
        for table in doc.tables:
            for row in table.rows:
                tables.append(' | '.join(c.text.strip() for c in row.cells))
        text = '\n'.join(paras + tables)
        print(f'✅ DOCX: {len(text)} chars')
        return text
    except Exception as e:
        print(f'❌ DOCX error: {e}')
        return ''


# ── Image OCR ─────────────────────────────────────────────
def extract_text_from_image(image_path: str) -> str:
    with open(image_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    img_type = 'image/png' if image_path.lower().endswith('.png') else 'image/jpeg'
    for model in [OCR_MODEL, 'google/gemini-flash-1.5']:
        try:
            print(f'🔄 OCR with {model}…')
            resp = llm_client.chat.completions.create(
                model=model,
                messages=[{'role': 'user', 'content': [
                    {'type': 'image_url', 'image_url': {'url': f'data:{img_type};base64,{b64}'}},
                    {'type': 'text',      'text': 'Extract all text from this CV/resume image. Preserve structure. Return only the text.'}
                ]}],
                max_tokens=3000
            )
            text = resp.choices[0].message.content
            print(f'✅ OCR: {len(text)} chars')
            return text
        except Exception as e:
            print(f'⚠️  OCR failed with {model}: {e}')
    return ''


# ── Unified file processor ─────────────────────────────────
def process_file(file_path: str) -> Tuple[str, Dict]:
    ext   = file_path.lower().split('.')[-1]
    links = {'linkedin': [], 'github': [], 'others': []}
    if ext == 'pdf':
        text  = extract_text_with_pymupdf(file_path).get('text', '')
        links = extract_links_from_pdf(file_path)
        return text, links
    elif ext in ('docx', 'doc'):
        return extract_text_from_docx(file_path), links
    elif ext in ('jpg', 'jpeg', 'png', 'bmp', 'tiff'):
        return extract_text_from_image(file_path), links
    else:
        raise ValueError(f'Unsupported file type: {ext}')


# ── LLM JSON parsers ──────────────────────────────────────

def _clean_json(raw: str) -> str:
    """Strip markdown fences and leading/trailing whitespace."""
    if '```json' in raw:
        raw = raw.split('```json')[1].split('```')[0]
    elif '```' in raw:
        parts = raw.split('```')
        raw = parts[1] if len(parts) >= 2 else raw
    raw = raw.strip()
    if not raw.startswith('{'):
        s, e = raw.find('{'), raw.rfind('}')
        if s != -1 and e != -1: raw = raw[s: e + 1]
    return raw


def parse_cv_with_docling_llm(file_path: str) -> Dict:
    """Docling + PyMuPDF → LLM (essentials only)."""
    print('🔄 Step 1: extracting text (Docling + PyMuPDF)…')
    text  = extract_text_with_pymupdf(file_path).get('text', '')
    links = extract_links_from_pdf(file_path)
    if links['linkedin']: text += '\n\n--- EXTRACTED LINKEDIN LINKS ---\n' + '\n'.join(links['linkedin'])
    if links['github']:   text += '\n\n--- EXTRACTED GITHUB LINKS ---\n'   + '\n'.join(links['github'])
    docling = extract_with_docling(file_path)
    if docling and docling.get('text'):
        text += '\n\n--- DOCLING OUTPUT ---\n\n' + docling['text']
    if not text:
        return _empty_cv_essentials()
    print(f'✅ Combined text: {len(text)} chars')
    if len(text) > 10000: text = text[:10000]; print('⚠️  Truncated to 10k chars')

    prompt = f"""You are an expert CV parser. Extract ONLY essential fields.

Return EXACTLY this JSON (no markdown, no preamble):
{{
  "full_name": "", "email": "", "phone": "",
  "linkedin": "full URL or empty", "github": "full URL or empty",
  "summary": "",
  "skills": {{"technical": [], "soft": []}},
  "experience": [
    {{"title": "", "company": "", "duration": "", "description": ""}}
  ]
}}

CV TEXT:
{text}"""

    for model in [PARSING_MODEL, PARSING_FALLBACK]:
        try:
            resp = llm_client.chat.completions.create(
                model=model,
                messages=[
                    {'role': 'system', 'content': 'CV parser. Return only valid JSON.'},
                    {'role': 'user',   'content': prompt}
                ],
                max_tokens=3000, temperature=0.1
            )
            result = resp.choices[0].message.content.strip()
            parsed = json.loads(_clean_json(result))
            print(f'✅ Docling+LLM parsing successful (model: {model})')
            return parsed
        except Exception as e:
            print(f'⚠️  {model} failed: {e}')
    return _empty_cv_essentials()


def _empty_cv_essentials() -> Dict:
    return {'full_name': '', 'email': '', 'phone': '', 'linkedin': '', 'github': '',
            'summary': '', 'skills': {'technical': [], 'soft': []}, 'experience': []}


def parse_cv_with_llm(cv_text: str) -> Dict:
    """PyMuPDF → LLM (full extraction)."""
    if len(cv_text) > 10000:
        cv_text = cv_text[:10000]; print('⚠️  Truncated to 10k chars')

    prompt = f"""You are an expert CV parser. Extract ALL information.

Rules:
- LinkedIn / GitHub: extract the full URL, not just the label text.
- If a field is missing, use empty string or empty array.
- Infer job_title from experience/skills if not explicit.
- Do NOT include degrees in certifications.
- Volunteer work may be inferred from context.

Return EXACTLY this JSON (no markdown):
{{
  "full_name":"","email":"","phone":"","linkedin":"","github":"",
  "job_title":"","summary":"",
  "education":[{{"degree":"","institution":"","location":"","start_date":"","end_date":"","gpa":""}}],
  "skills":{{"technical":[],"soft":[],"tools":[]}},
  "experience":[{{"title":"","company":"","location":"","employment_type":"",
                  "start_date":"","end_date":"","duration":"",
                  "responsibilities":[],"achievements":[],"technologies":[]}}],
  "certifications":[{{"name":"","issuer":"","issue_date":""}}],
  "projects":[{{"name":"","description":"","technologies":[],"date":""}}],
  "languages":[{{"language":"","proficiency":""}}],
  "awards":[{{"title":"","issuer":"","description":""}}],
  "volunteer":[{{"role":"","organization":"","duration":"","description":""}}]
}}

CV TEXT:
{cv_text}"""

    try:
        print(f'🤖 Parsing CV with {PARSING_MODEL}…')
        resp = llm_client.chat.completions.create(
            model=PARSING_MODEL,
            messages=[
                {'role': 'system', 'content': 'Professional CV parser. Return only valid JSON.'},
                {'role': 'user',   'content': prompt}
            ],
            max_tokens=4000, temperature=0.1
        )
        result = resp.choices[0].message.content.strip()
        parsed = json.loads(_clean_json(result))
        print('✅ CV parsing successful.')
        return parsed
    except Exception as e:
        print(f'❌ LLM CV parsing error: {e}')
        return {'full_name': '', 'email': '', 'phone': '', 'linkedin': '', 'github': '',
                'job_title': '', 'summary': '', 'education': [],
                'skills': {'technical': [], 'soft': [], 'tools': []},
                'experience': [], 'certifications': [], 'projects': [],
                'languages': [], 'awards': [], 'volunteer': []}


print('✅ CV parsing helpers defined.')

✅ CV parsing helpers defined.


In [ ]:
# ============================================================
# CELL 7 — ATS ANALYSIS + IMPROVEMENT GENERATOR
# ============================================================

def analyze_ats_with_llm(parsed_cv: Dict, cv_text: str) -> Dict:
    cv_summary = json.dumps(parsed_cv, indent=2)
    prompt = f"""You are a STRICT ATS analyst with 15+ years of experience.
Most CVs score 50-75. Only exceptional CVs exceed 85.

Score each category 0-100 using these benchmarks:
- formatting:              100=perfect ATS-friendly, 60-79=readable with issues, <60=ATS may fail
- content_quality:         100=every bullet has action verb + metric, 60-79=generic descriptions
- keyword_optimization:    100=15+ industry keywords, 60-79=4-7 keywords
- experience_presentation: 100=clear progression + quantified achievements
- skills_relevance:        100=15+ in-demand skills, 60-79=5-9 skills
- completeness:            100=all sections present, deduct for each missing section
- professionalism:         100=zero errors, professional email/LinkedIn

overall_score = average of all 7 categories.

CV DATA:
{cv_summary}

Return EXACTLY this JSON (no markdown):
{{
  "overall_score": 0,
  "grade": "",
  "summary_feedback": "",
  "scores": {{"formatting":0,"content_quality":0,"keyword_optimization":0,
              "experience_presentation":0,"skills_relevance":0,"completeness":0,"professionalism":0}},
  "strengths": [],
  "weaknesses": [],
  "critical_issues": [],
  "improvement_suggestions": [
    {{"category":"","priority":"","suggestion":"","impact":""}}
  ],
  "missing_elements": [],
  "keywords_analysis": {{"strong_keywords":[],"missing_keywords":[],"suggested_keywords":[]}},
  "recruiter_perspective": "",
  "next_steps": []
}}"""

    try:
        print(f'🤖 ATS analysis with {ATS_MODEL}…')
        resp = llm_client.chat.completions.create(
            model=ATS_MODEL,
            messages=[
                {'role': 'system', 'content': 'Strict ATS analyst. Return only valid JSON.'},
                {'role': 'user',   'content': prompt}
            ],
            max_tokens=3000, temperature=0.2
        )
        ats = json.loads(_clean_json(resp.choices[0].message.content.strip()))
        # Recalibrate if suspiciously high
        if ats.get('overall_score', 0) > 95:
            scores = ats.get('scores', {})
            ats['overall_score'] = round(sum(scores.values()) / len(scores)) if scores else 70
        print(f'✅ ATS score: {ats["overall_score"]}/100')
        return ats
    except Exception as e:
        print(f'⚠️  ATS analysis error: {e}')
        return {'overall_score': 65, 'grade': 'D+',
                'summary_feedback': 'CV needs improvements.',
                'scores': {'formatting':60,'content_quality':55,'keyword_optimization':50,
                           'experience_presentation':70,'skills_relevance':65,
                           'completeness':75,'professionalism':70},
                'strengths': ['CV structure is present'],
                'weaknesses': ['Lacks quantified achievements', 'Missing keywords'],
                'critical_issues': ['No measurable results in experience section'],
                'improvement_suggestions': [{'category':'Content','priority':'Critical',
                    'suggestion':"Add metrics to every achievement (e.g., 'Increased X by Y%')",
                    'impact':'+15 points'}],
                'missing_elements': ['Quantified achievements'],
                'keywords_analysis': {'strong_keywords':[],'missing_keywords':['Industry-specific terms'],
                                      'suggested_keywords':['Add role-specific keywords']},
                'recruiter_perspective': 'CV shows potential but needs quantification and keyword optimisation.',
                'next_steps': ['Add metrics', 'Optimise keywords', 'Fix formatting']}


def generate_improvements_with_llm(parsed_cv: Dict, ats_result: Dict) -> Dict:
    prompt = f"""You are a professional CV consultant.

CV: {json.dumps(parsed_cv, indent=2)}
ATS ANALYSIS: {json.dumps(ats_result, indent=2)}

Return EXACTLY this JSON (no markdown):
{{
  "priority_actions": [{{"action":"","reason":"","how_to":"","priority":""}}],
  "content_rewrites": {{
    "professional_summary": "",
    "experience_improvements": [{{"current":"","improved":"","why_better":""}}]
  }},
  "skills_strategy": {{"skills_to_add":[],"skills_to_emphasize":[],
                       "skills_to_remove":[],"how_to_demonstrate":[]}},
  "formatting_checklist": [],
  "keyword_strategy": {{"must_add":[],"contextual_usage":[{{"keyword":"","where":"","example":""}}]}},
  "achievement_framework": {{"suggested_achievements":[],"quantification_tips":[]}},
  "30_day_plan": [{{"week":1,"tasks":[],"goal":""}}]
}}"""

    try:
        print(f'🤖 Generating improvement plan with {SCORING_MODEL}…')
        resp = llm_client.chat.completions.create(
            model=SCORING_MODEL,
            messages=[
                {'role': 'system', 'content': 'CV improvement expert. Return only valid JSON.'},
                {'role': 'user',   'content': prompt}
            ],
            max_tokens=3000, temperature=0.4
        )
        result = json.loads(_clean_json(resp.choices[0].message.content.strip()))
        print('✅ Improvement plan generated.')
        return result
    except Exception as e:
        print(f'⚠️  Improvement generation error: {e}')
        return {'priority_actions': [{'action': 'Review ATS feedback', 'reason': 'Address identified issues',
                                      'how_to': 'Follow the suggestions provided', 'priority': 'High'}],
                'content_rewrites': {}, 'skills_strategy': {}, 'formatting_checklist': [],
                'keyword_strategy': {}, 'achievement_framework': {}, '30_day_plan': []}


print('✅ ATS helpers defined.')

✅ ATS helpers defined.


In [ ]:
# ============================================================
# CELL 8 — JOB MATCHER (queries real ChromaDB jobs)
# ============================================================

class JobMatcher:
    """Matches a parsed CV against jobs stored in jobs_col."""

    WEIGHTS = {
        'job_title':      0.25,
        'skills':         0.30,
        'experience':     0.25,
        'summary':        0.10,
        'projects':       0.05,
        'certifications': 0.05,
    }

    def __init__(self):
        self.model = cv_embedding_model

    # ── CV component builder ─────────────────────────────
    def _cv_components(self, cv: Dict) -> Dict[str, str]:
        c = {}
        c['job_title'] = f"Target Role: {cv.get('job_title', '')}"
        skills_data = cv.get('skills', {})
        all_skills  = []
        if isinstance(skills_data, dict):
            for lst in skills_data.values():
                if isinstance(lst, list): all_skills.extend(lst)
        c['skills'] = f"Skills: {', '.join(all_skills)}"
        exp_parts = []
        for e in cv.get('experience', []):
            parts = [e.get('title', ''), f"at {e.get('company', '')}"]
            if isinstance(e.get('responsibilities'), list): parts += e['responsibilities']
            if isinstance(e.get('achievements'),    list): parts += e['achievements']
            if e.get('description'): parts.append(e['description'])
            exp_parts.append(' '.join(parts))
        c['experience']     = ' '.join(exp_parts)
        c['summary']        = cv.get('summary', '')
        c['projects']       = ' '.join(p.get('description', '') for p in cv.get('projects', []))
        c['certifications'] = ' '.join(f"Certified in {cert.get('name', '')}" for cert in cv.get('certifications', []))
        return c

    # ── Job component builder ─────────────────────────────
    def _job_components(self, job: Dict) -> Dict[str, str]:
        skills = job.get('required_skills', [])
        if isinstance(skills, list): skills = ', '.join(skills)
        return {
            'job_title':      job.get('title', ''),
            'skills':         skills,
            'experience':     job.get('description', ''),
            'summary':        job.get('description', ''),
            'projects':       '',
            'certifications': job.get('preferred_qualifications', '') or '',
        }

    def _weighted_sim(self, cv_c: Dict, job_c: Dict) -> float:
        total, weight_sum = 0.0, 0.0
        for key, w in self.WEIGHTS.items():
            cv_t, job_t = cv_c.get(key, ''), job_c.get(key, '')
            if cv_t and job_t:
                sim = cosine_similarity(
                    self.model.encode([cv_t]),
                    self.model.encode([job_t])
                )[0][0]
                total      += sim * w
                weight_sum += w
        return (total / weight_sum * 100) if weight_sum > 0 else 0.0

    def _llm_match(self, cv: Dict, job: Dict) -> Optional[Dict]:
        prompt = f"""You are a recruitment matching expert.

CANDIDATE CV: {json.dumps(cv, indent=2)}
JOB POSITION: {json.dumps(job, indent=2)}

Return EXACTLY this JSON (no markdown):
{{
  "match_score": 0,
  "match_level": "Excellent/Good/Fair/Poor",
  "recommendation": "",
  "detailed_scores": {{"skills_match":0,"experience_match":0,"education_match":0,
                       "cultural_fit":0,"career_trajectory":0}},
  "matched_requirements": [],
  "missing_requirements": [{{"requirement":"","importance":"","can_learn":true,"time_to_acquire":""}}],
  "strengths_for_role": [],
  "concerns": [],
  "differentiators": [],
  "interview_talking_points": [],
  "likelihood_of_offer": "High/Medium/Low",
  "application_strategy": ""
}}"""
        try:
            resp = llm_client.chat.completions.create(
                model=SCORING_MODEL,
                messages=[
                    {'role': 'system', 'content': 'Recruitment expert. Return only valid JSON.'},
                    {'role': 'user',   'content': prompt}
                ],
                max_tokens=2000, temperature=0.3
            )
            return json.loads(_clean_json(resp.choices[0].message.content.strip()))
        except Exception as e:
            print(f"⚠️  LLM match error for '{job.get('title')}':: {e}")
            return None

    def match_jobs_from_db(self, parsed_cv: Dict, n_results: int = 5) -> List[Dict]:
        """
        Query the shared ChromaDB jobs collection for the most relevant jobs,
        then run LLM analysis on the top hits.
        """
        total_jobs = jobs_col.count()
        if total_jobs == 0:
            print('⚠️  No scraped jobs in database. Run the scraper first (Cell 9).')
            return []

        # Build a query string from the CV
        cv_components = self._cv_components(parsed_cv)
        query_text = ' '.join(filter(None, [
            cv_components.get('job_title', ''),
            cv_components.get('skills', ''),
            cv_components.get('experience', '')[:500],
        ]))

        n_fetch = min(n_results * 3, total_jobs)   # over-fetch, then re-rank
        print(f'🔍 Querying {total_jobs} scraped jobs for top {n_fetch} candidates…')
        db_results = jobs_col.query(
            query_texts=[query_text],
            n_results=n_fetch
        )

        candidates = []
        for i in range(len(db_results['ids'][0])):
            meta = db_results['metadatas'][0][i]
            try:
                job_detail = json.loads(meta.get('json_detailed', '{}'))
            except Exception:
                job_detail = {}
            job_doc = {
                'title':           meta.get('title', 'N/A'),
                'company':         meta.get('company', 'N/A'),
                'location':        meta.get('location', 'N/A'),
                'source':          meta.get('source', 'N/A'),
                'apply_link':      meta.get('apply_link', meta.get('job_page_link', '#')),
                'description':     job_detail.get('description', ''),
                'required_skills': job_detail.get('skills', []),
                'experience_level': meta.get('experience_level', ''),
                'employment_type': meta.get('employment_type', ''),
            }
            # Weighted semantic similarity
            sim = self._weighted_sim(cv_components, self._job_components(job_doc))
            candidates.append((sim, job_doc))

        # Sort by semantic score, keep top n_results for LLM deep-dive
        candidates.sort(key=lambda x: x[0], reverse=True)
        top_n = candidates[:n_results]

        results = []
        for sim, job in top_n:
            print(f"  🔎 LLM analysis for '{job['title']}' at {job['company']}…")
            llm_analysis = self._llm_match(parsed_cv, job)
            entry = {
                'job_title':          job['title'],
                'company':            job['company'],
                'location':           job['location'],
                'source':             job['source'],
                'apply_link':         job['apply_link'],
                'semantic_similarity': round(sim, 2),
            }
            if llm_analysis:
                entry.update(llm_analysis)
            else:
                entry.update({'match_score': round(sim, 2),
                              'match_level': self._match_level(sim),
                              'recommendation': 'Based on semantic similarity only.'})
            results.append(entry)

        results.sort(key=lambda x: x.get('match_score', 0), reverse=True)
        print('✅ Job matching complete.')
        return results

    @staticmethod
    def _match_level(score: float) -> str:
        if score >= 85: return '🟢 Excellent'
        if score >= 70: return '🟡 Good'
        if score >= 55: return '🟠 Fair'
        return '🔴 Poor'


print('✅ JobMatcher defined (queries real scraped jobs).')

✅ JobMatcher defined (queries real scraped jobs).


In [ ]:
# ============================================================
# CELL 9 — RUN SCRAPER (optional — skip if DB already has data)
# ============================================================
#
# This cell scrapes Wuzzuf + LinkedIn for every TARGET_CATEGORY
# and stores results in the shared `job_listings` ChromaDB collection.
#
# ⚠️  Full run takes ~1-2 hours.  To run only a subset, slice
#     TARGET_CATEGORIES before calling run_scraper().

current_count = jobs_col.count()
print(f'Current job count in DB: {current_count}')

if current_count == 0:
    print('DB is empty — starting full scrape…')
    def _run_scraper_thread():
        """Run the async scraper inside a fresh event loop thread (Colab-safe)."""
        if sys.platform == 'win32':
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(run_scraper())
        finally:
            loop.close()

    import threading
    t = threading.Thread(target=_run_scraper_thread)
    t.start()
    t.join()
    print(f'✅ Scraping done. DB now has {jobs_col.count()} jobs.')
else:
    print(f'✅ DB already has {current_count} jobs — skipping scraper.')
    print('   To force a fresh scrape, call run_scraper() manually.')

Current job count in DB: 1734
✅ DB already has 1734 jobs — skipping scraper.
   To force a fresh scrape, call run_scraper() manually.


In [ ]:
# ============================================================
# CELL 10 — INTERACTIVE CV ANALYSIS
# ============================================================

def run_cv_analysis(parsing_mode: str = 'llm', n_job_matches: int = 5):
    """
    Interactive CV analysis.
    parsing_mode: 'llm' (full details) | 'docling' (essentials only)
    n_job_matches: how many real DB jobs to compare against
    """
    print('=' * 70)
    print('🎯 JOBLENS — AI-POWERED CV ANALYSIS')
    print(f'📋 Mode: {parsing_mode.upper()}')
    print('=' * 70)

    if IN_COLAB:
        print('\n📤 Upload your CV (PDF, DOCX or Image)…')
        uploaded = files.upload()
        if not uploaded:
            print('❌ No file uploaded.')
            return
        file_name = list(uploaded.keys())[0]
    else:
        file_name = input('Enter the path to your CV file: ').strip()

    print(f'\n✅ File: {file_name}')

    # ── Parse CV ──────────────────────────────────────────
    print('\n' + '─' * 70)
    print('STEP 1: PARSE CV')
    if parsing_mode == 'docling':
        parsed_cv = parse_cv_with_docling_llm(file_name)
        cv_text, _ = process_file(file_name)
    else:
        cv_text, _ = process_file(file_name)
        if not cv_text:
            print('❌ Could not extract text.')
            return
        parsed_cv = parse_cv_with_llm(cv_text)

    print('\n📋 Parsed CV:')
    print(json.dumps(parsed_cv, indent=2, ensure_ascii=False))

    # ── AI Content Detection ──────────────────────────────
    print('\n' + '─' * 70)
    print('STEP 2: AI CONTENT DETECTION')
    text_for_ai = cv_text if isinstance(cv_text, str) else (cv_text[0] if cv_text else '')
    ai_res = ai_detector.analyze_text(text_for_ai)
    print(f'  📊 AI Probability : {ai_res["ai_probability_score"]}/100')
    print(f'  🎯 Verdict        : {ai_res["verdict"]}')
    print(f'  📈 Perplexity     : {ai_res["avg_perplexity"]}')
    print(f'  💥 Burstiness     : {ai_res["burstiness_score"]}')

    # ── ATS Analysis ──────────────────────────────────────
    print('\n' + '─' * 70)
    print('STEP 3: ATS ANALYSIS')
    ats_res = analyze_ats_with_llm(parsed_cv, text_for_ai)
    print(f'\n📊 ATS Score: {ats_res["overall_score"]}/100  ({ats_res["grade"]})')
    print(f'💬 {ats_res["summary_feedback"]}')
    print('\n📈 Scores by category:')
    for k, v in ats_res.get('scores', {}).items():
        print(f'  • {k.replace("_", " ").title()}: {v}/100')
    print('\n✅ Strengths:')
    for s in ats_res.get('strengths', [])[:3]: print(f'  • {s}')
    print('\n⚠️  Weaknesses:')
    for w in ats_res.get('weaknesses', [])[:3]: print(f'  • {w}')
    if ats_res.get('critical_issues'):
        print('\n🚨 Critical Issues:')
        for i in ats_res['critical_issues']: print(f'  • {i}')

    # ── Improvements ──────────────────────────────────────
    print('\n' + '─' * 70)
    print('STEP 4: IMPROVEMENT PLAN')
    improvements = generate_improvements_with_llm(parsed_cv, ats_res)
    print('\n🎯 Priority actions:')
    for a in improvements.get('priority_actions', [])[:3]:
        print(f'  [{a.get("priority")}] {a.get("action")}')
        print(f'   Why: {a.get("reason")}')
    if improvements.get('content_rewrites', {}).get('professional_summary'):
        print(f'\n📝 Suggested summary:\n  {improvements["content_rewrites"]["professional_summary"]}')

    # ── Job Matching from real DB ──────────────────────────
    print('\n' + '─' * 70)
    print(f'STEP 5: JOB MATCHING (against {jobs_col.count()} scraped jobs)')
    matcher = JobMatcher()
    matches = matcher.match_jobs_from_db(parsed_cv, n_results=n_job_matches)

    print('\n🎯 TOP JOB MATCHES:\n')
    for i, m in enumerate(matches, 1):
        print(f'{i}. {m.get("job_title")} at {m.get("company")}')
        print(f'   📍 {m.get("location")}  |  Source: {m.get("source")}')
        print(f'   🔗 {m.get("apply_link", "#")}')
        if m.get('detailed_scores'):
            print('   📊 Breakdown:')
            for k, v in m['detailed_scores'].items():
                print(f'      • {k.replace("_", " ").title()}: {v}/100')
        if m.get('matched_requirements'):
            print(f'   ✅ Matched ({len(m["matched_requirements"])}):', ', '.join(m['matched_requirements'][:3]))
        if m.get('missing_requirements'):
            print(f'   ❌ Missing ({len(m["missing_requirements"])}):')
            for miss in m['missing_requirements'][:3]:
                if isinstance(miss, dict):
                    print(f'      • {miss.get("requirement")} ({miss.get("importance")})')
                else:
                    print(f'      • {miss}')
        print(f'   💡 {m.get("recommendation", "")}\n')

    print('=' * 70)
    print('✅ ANALYSIS COMPLETE!')
    print('=' * 70)

    return {'parsed_cv': parsed_cv, 'ats_result': ats_res,
            'improvements': improvements, 'job_matches': matches}


# ── Mode selector ──────────────────────────────────────────
def run_interactive():
    print('\n📋 Choose parsing mode:')
    print('  1 — Docling mode  (essentials only, faster)')
    print('  2 — LLM mode      (full details, recommended)')
    choice = input('\n👉 Select (1 or 2): ').strip()
    mode   = 'docling' if choice == '1' else 'llm'
    print(f'✅ Mode: {mode}')
    return run_cv_analysis(parsing_mode=mode)


# ── Run ───────────────────────────────────────────────────
result = run_interactive()


📋 Choose parsing mode:
  1 — Docling mode  (essentials only, faster)
  2 — LLM mode      (full details, recommended)

👉 Select (1 or 2): 1
✅ Mode: docling
🎯 JOBLENS — AI-POWERED CV ANALYSIS
📋 Mode: DOCLING

📤 Upload your CV (PDF, DOCX or Image)…


[INFO] 2026-03-26 15:06:46,699 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-26 15:06:46,718 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-26 15:06:46,719 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx


Saving Mohamed_Oraby_CV.pdf to Mohamed_Oraby_CV.pdf

✅ File: Mohamed_Oraby_CV.pdf

──────────────────────────────────────────────────────────────────────
STEP 1: PARSE CV
🔄 Step 1: extracting text (Docling + PyMuPDF)…
✅ PyMuPDF: 6498 chars from 2 page(s)
🔗 Links found — LinkedIn: 1, GitHub: 5
🔄 Processing with Docling…


[INFO] 2026-03-26 15:06:46,848 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-26 15:06:46,854 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-26 15:06:46,854 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-26 15:06:46,914 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-26 15:06:46,955 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
[INFO] 2026-03-26 15:06:46,956 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx


✅ Docling: 8017 chars
✅ Combined text: 14920 chars
⚠️  Truncated to 10k chars
✅ Docling+LLM parsing successful (model: meta-llama/llama-3.3-70b-instruct)
✅ PyMuPDF: 6498 chars from 2 page(s)
🔗 Links found — LinkedIn: 1, GitHub: 5

📋 Parsed CV:
{
  "full_name": "Mohamed Oraby",
  "email": "mohamed.khalid.gamal@gmail.com",
  "phone": "+201211099674",
  "linkedin": "https://www.linkedin.com/in/mohamed-khalid-gamal",
  "github": "https://github.com/mohamed-khalid-gamal",
  "summary": "Motivated Computer Science senior at Ain Shams University with hands-on experience in full-stack development, software engineering, and data-driven applications.",
  "skills": {
    "technical": [
      "ASP .NET Core",
      "C#",
      "JS",
      "NodeJs",
      "PHP",
      "Angular",
      "SQL Server",
      "Entity Framework",
      "RESTful APIs",
      "Microservices",
      "Clean Architecture"
    ],
    "soft": []
  },
  "experience": [
    {
      "title": "Full Stack .NET & Angular Engineer",
  

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  📊 AI Probability : 0/100
  🎯 Verdict        : 👤 Likely Human Written
  📈 Perplexity     : 532.14
  💥 Burstiness     : 696.39

──────────────────────────────────────────────────────────────────────
STEP 3: ATS ANALYSIS
🤖 ATS analysis with meta-llama/llama-3.3-70b-instruct…
✅ ATS score: 73/100

📊 ATS Score: 73/100  (Average)
💬 The CV is well-structured and easy to read, but lacks quantifiable achievements and industry keywords.

📈 Scores by category:
  • Formatting: 90/100
  • Content Quality: 60/100
  • Keyword Optimization: 40/100
  • Experience Presentation: 80/100
  • Skills Relevance: 80/100
  • Completeness: 90/100
  • Professionalism: 90/100

✅ Strengths:
  • Clear and concise summary
  • Relevant technical skills
  • Variety of experience in different roles

⚠️  Weaknesses:
  • Lack of quantifiable achievements
  • Insufficient industry keywords
  • No clear career progression

🚨 Critical Issues:
  • Email address does not match LinkedIn profile
  • No mention of relevant certi

In [ ]:
# ============================================================
# CELL 11 — UNIFIED FASTAPI SERVER
# ============================================================
#
# All endpoints from both original notebooks in one app,
# all sharing the single chroma_client / jobs_col / candidates_col.

scheduler = AsyncIOScheduler()

async def scheduled_scrape():
    print('⏰ [Scheduler] Starting automated scrape…')
    try:
        await run_scraper()
        print('⏰ [Scheduler] Scrape complete.')
    except Exception as e:
        print(f'❌ [Scheduler] Error: {e}')


@asynccontextmanager
async def lifespan(app: FastAPI):
    print('⏳ Starting APScheduler (every 6 hours)…')
    scheduler.add_job(scheduled_scrape, 'interval', hours=6)
    scheduler.start()
    yield
    print('🛑 Shutting down scheduler…')
    scheduler.shutdown()


app = FastAPI(title='JobLens Unified API', version='2.0.0', lifespan=lifespan)

# ── Pydantic models ──────────────────────────────────────

class CVParseTextRequest(BaseModel):
    resume_text: str

class ATSRequest(BaseModel):
    resume_text: str
    job_description: Optional[str] = None

class CandidateEmbeddingRequest(BaseModel):
    candidate_id: int
    profile_data: Dict[str, Any]

class JobEmbeddingRequest(BaseModel):
    job_id:   int
    job_data: Dict[str, Any]

class StandardResponse(BaseModel):
    success: bool
    data:    Optional[Dict[str, Any]] = None
    message: Optional[str]            = None

class ListResponse(BaseModel):
    success: bool
    data:    Optional[List[Dict[str, Any]]] = None
    message: Optional[str]                 = None


# ─────────────────────────────────────────────────────────
# GROUP A  — Scraping endpoints
# ─────────────────────────────────────────────────────────

@app.get('/api/scraping/jobs', summary='Get scraped jobs with optional keyword/location filter')
async def get_scraped_jobs(
    keyword:  Optional[str] = Query(None),
    location: Optional[str] = Query(None),
    limit:    int           = 50
):
    try:
        where = {'location': location} if location else None
        if keyword:
            q_vec = scraper_embed_model.encode(keyword).tolist()
            res   = jobs_col.query(query_embeddings=[q_vec], n_results=limit, where=where)
            ids, metas = res['ids'][0], res['metadatas'][0]
        else:
            res   = jobs_col.get(limit=limit, where=where)
            ids, metas = res['ids'], res['metadatas']

        data = []
        for i, m in enumerate(metas):
            detail = {}
            try: detail = json.loads(m.get('json_detailed', '{}'))
            except Exception: pass
            data.append({'db_id': ids[i], 'company': m.get('company'), 'title': m.get('title'),
                         'location': m.get('location'), **detail})
        return {'success': True, 'count': len(ids), 'data': data}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post('/api/scraping/trigger', status_code=202,
          summary='Manually trigger a background scrape')
async def trigger_scrape(bt: BackgroundTasks):
    bt.add_task(run_scraper)
    return {'success': True, 'message': 'Scraping job queued.'}


# ─────────────────────────────────────────────────────────
# GROUP B  — CV parsing endpoints
# ─────────────────────────────────────────────────────────

@app.post('/api/cv/parse', response_model=StandardResponse,
          summary='Upload a CV file and get structured JSON')
async def parse_cv_file(file: UploadFile = File(...)):
    tmp = None
    try:
        ext = file.filename.split('.')[-1]
        with tempfile.NamedTemporaryFile(delete=False, suffix=f'.{ext}') as tf:
            tf.write(await file.read())
            tmp = tf.name
        cv_text, _ = process_file(tmp)
        if not cv_text:
            return StandardResponse(success=False, message='Could not extract text.')
        parsed = parse_cv_with_llm(cv_text)
        return StandardResponse(success=True, data=parsed)
    except Exception as e:
        return StandardResponse(success=False, message=str(e))
    finally:
        if tmp and os.path.exists(tmp): os.remove(tmp)


@app.post('/api/cv/parse-text', response_model=StandardResponse,
          summary='Parse raw CV text')
async def parse_cv_text(req: CVParseTextRequest):
    try:
        return StandardResponse(success=True, data=parse_cv_with_llm(req.resume_text))
    except Exception as e:
        return StandardResponse(success=False, message=str(e))


@app.post('/api/cv/ats-score', response_model=StandardResponse,
          summary='Get ATS compatibility score for a CV')
async def ats_score(req: ATSRequest):
    try:
        parsed = parse_cv_with_llm(req.resume_text)
        ats    = analyze_ats_with_llm(parsed, req.resume_text)
        return StandardResponse(success=True, data=ats)
    except Exception as e:
        return StandardResponse(success=False, message=str(e))


# ─────────────────────────────────────────────────────────
# GROUP C  — Candidate embedding CRUD
# ─────────────────────────────────────────────────────────

@app.post('/api/embeddings/candidate', response_model=StandardResponse)
async def create_candidate_embedding(req: CandidateEmbeddingRequest):
    try:
        doc = json.dumps(req.profile_data)
        candidates_col.add(
            documents=[doc],
            metadatas=[{'candidate_id': req.candidate_id}],
            ids=[str(req.candidate_id)]
        )
        return StandardResponse(success=True, message='Candidate embedding created.',
                                data={'embedding_id': req.candidate_id})
    except Exception as e:
        return StandardResponse(success=False, message=str(e))


@app.put('/api/embeddings/candidate/{candidate_id}', response_model=StandardResponse)
async def update_candidate_embedding(candidate_id: str, req: CandidateEmbeddingRequest):
    try:
        candidates_col.update(
            ids=[candidate_id],
            documents=[json.dumps(req.profile_data)],
            metadatas=[{'candidate_id': req.candidate_id}]
        )
        return StandardResponse(success=True, message='Candidate embedding updated.')
    except Exception as e:
        return StandardResponse(success=False, message=str(e))


@app.delete('/api/embeddings/candidate/{candidate_id}', response_model=StandardResponse)
async def delete_candidate_embedding(candidate_id: str):
    try:
        candidates_col.delete(ids=[candidate_id])
        return StandardResponse(success=True, message='Candidate embedding deleted.')
    except Exception as e:
        return StandardResponse(success=False, message=str(e))


# ─────────────────────────────────────────────────────────
# GROUP D  — Job embedding CRUD (manual / external jobs)
# ─────────────────────────────────────────────────────────

@app.post('/api/embeddings/job', response_model=StandardResponse)
async def create_job_embedding(req: JobEmbeddingRequest):
    try:
        jobs_col.add(
            documents=[json.dumps(req.job_data)],
            metadatas=[{'job_id': req.job_id, 'source': 'Internal API',
                        'title':   req.job_data.get('title', ''),
                        'company': req.job_data.get('company', ''),
                        'location':req.job_data.get('location', ''),
                        'json_detailed': json.dumps(req.job_data)}],
            ids=[f'job_{req.job_id}']
        )
        return StandardResponse(success=True, message='Job embedding created.',
                                data={'embedding_id': f'job_{req.job_id}'})
    except Exception as e:
        return StandardResponse(success=False, message=str(e))


@app.put('/api/embeddings/job/{job_id}', response_model=StandardResponse)
async def update_job_embedding(job_id: str, req: JobEmbeddingRequest):
    try:
        jobs_col.update(
            ids=[job_id],
            documents=[json.dumps(req.job_data)],
            metadatas=[{'job_id': req.job_id, 'json_detailed': json.dumps(req.job_data)}]
        )
        return StandardResponse(success=True, message='Job embedding updated.')
    except Exception as e:
        return StandardResponse(success=False, message=str(e))


@app.delete('/api/embeddings/job/{job_id}', response_model=StandardResponse)
async def delete_job_embedding(job_id: str):
    try:
        jobs_col.delete(ids=[job_id])
        return StandardResponse(success=True, message='Job embedding deleted.')
    except Exception as e:
        return StandardResponse(success=False, message=str(e))


# ─────────────────────────────────────────────────────────
# GROUP E  — Recommendation endpoints
# ─────────────────────────────────────────────────────────

@app.get('/api/recommendations/jobs/{candidate_id}', response_model=ListResponse,
         summary='Get top job recommendations for a stored candidate')
async def recommend_jobs(candidate_id: int, limit: int = 10):
    try:
        cand = candidates_col.get(ids=[str(candidate_id)])
        if not cand['documents']:
            return ListResponse(success=False, message='Candidate not found.')
        res = jobs_col.query(query_texts=[cand['documents'][0]], n_results=limit)
        matches = [
            {'job_id': res['ids'][0][i],
             'match_score': round(max(0.0, 1.0 - res['distances'][0][i]), 2),
             'job_preview': json.loads(res['documents'][0][i])}
            for i in range(len(res['ids'][0]))
        ]
        return ListResponse(success=True, data=matches)
    except Exception as e:
        return ListResponse(success=False, message=str(e))


@app.get('/api/recommendations/candidates/{job_id}', response_model=ListResponse,
         summary='Get ranked candidates for a given job')
async def recommend_candidates(job_id: str, limit: int = 50, min_score: float = 0.3):
    try:
        job = jobs_col.get(ids=[str(job_id)])
        if not job['documents']:
            return ListResponse(success=False, message='Job not found.')
        res = candidates_col.query(query_texts=[job['documents'][0]], n_results=limit)
        matches = []
        if res['distances']:
            for i in range(len(res['ids'][0])):
                score = max(0.0, 1.0 - res['distances'][0][i])
                if score >= min_score:
                    matches.append({'candidate_id': res['ids'][0][i], 'score': round(score, 2),
                                    'candidate_preview': json.loads(res['documents'][0][i])})
        matches.sort(key=lambda x: x['score'], reverse=True)
        return ListResponse(success=True, data=matches)
    except Exception as e:
        return ListResponse(success=False, message=str(e))


# ─────────────────────────────────────────────────────────
# Start server in background thread + wait until it is ready
# ─────────────────────────────────────────────────────────
import time, socket

SERVER_HOST = '127.0.0.1'
SERVER_PORT = 8000

import asyncio
def _start_server():
    # Use 0.0.0.0 so ngrok can reach it even when Colab's loopback
    # behaves unexpectedly; we still expose only on localhost via ngrok.
    config = uvicorn.Config(app, host='0.0.0.0', port=SERVER_PORT,
                            log_level='warning')
    server = uvicorn.Server(config)

    # Create a new event loop for this background thread
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    # Run the async serve() method inside our new loop
    loop.run_until_complete(server.serve())

def _wait_for_port(host: str, port: int, timeout: float = 30.0):
    """Block until the TCP port is open or timeout expires."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.5)
    return False

# Kill any previous server occupying the port (re-run safety)
import subprocess
subprocess.run(['fuser', '-k', f'{SERVER_PORT}/tcp'],
               capture_output=True)   # silently fails on non-Linux — that's fine
time.sleep(1)

api_thread = threading.Thread(target=_start_server, daemon=True)
api_thread.start()

print(f'⏳ Waiting for FastAPI to bind on port {SERVER_PORT}…')
if _wait_for_port(SERVER_HOST, SERVER_PORT, timeout=30):
    print('\n' + '=' * 60)
    print('🚀 JobLens Unified API is running!')
    print(f'📚 Swagger UI → http://{SERVER_HOST}:{SERVER_PORT}/docs')
    print('=' * 60)
    print('\nEndpoints:')
    print('  GET  /api/scraping/jobs              — browse scraped jobs')
    print('  POST /api/scraping/trigger           — manual scrape')
    print('  POST /api/cv/parse                   — parse uploaded CV file')
    print('  POST /api/cv/parse-text              — parse raw CV text')
    print('  POST /api/cv/ats-score               — get ATS score')
    print('  POST /api/embeddings/candidate       — store candidate embedding')
    print('  PUT  /api/embeddings/candidate/{id}  — update candidate')
    print('  DEL  /api/embeddings/candidate/{id}  — delete candidate')
    print('  POST /api/embeddings/job             — store manual job')
    print('  PUT  /api/embeddings/job/{id}        — update job')
    print('  DEL  /api/embeddings/job/{id}        — delete job')
    print('  GET  /api/recommendations/jobs/{cid} — jobs for a candidate')
    print('  GET  /api/recommendations/candidates/{jid} — candidates for a job')
else:
    print('❌ Server did NOT start within 30 s — check the thread output above.')

⏳ Waiting for FastAPI to bind on port 8000…
⏳ Starting APScheduler (every 6 hours)…

🚀 JobLens Unified API is running!
📚 Swagger UI → http://127.0.0.1:8000/docs

Endpoints:
  GET  /api/scraping/jobs              — browse scraped jobs
  POST /api/scraping/trigger           — manual scrape
  POST /api/cv/parse                   — parse uploaded CV file
  POST /api/cv/parse-text              — parse raw CV text
  POST /api/cv/ats-score               — get ATS score
  POST /api/embeddings/candidate       — store candidate embedding
  PUT  /api/embeddings/candidate/{id}  — update candidate
  DEL  /api/embeddings/candidate/{id}  — delete candidate
  POST /api/embeddings/job             — store manual job
  PUT  /api/embeddings/job/{id}        — update job
  DEL  /api/embeddings/job/{id}        — delete job
  GET  /api/recommendations/jobs/{cid} — jobs for a candidate
  GET  /api/recommendations/candidates/{jid} — candidates for a job


In [ ]:
# ============================================================
# CELL 12 — EXPOSE VIA NGROK (public URL)
# ============================================================
#
# Prerequisites:
#   • Cell 11 must show '🚀 JobLens Unified API is running!'
#   • NGROK_AUTH_TOKEN must be set in Colab Secrets

import time, requests, os
from pyngrok import ngrok, conf
from google.colab import userdata  # <-- ADDED THIS IMPORT

try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
except Exception:
    NGROK_AUTH_TOKEN = os.getenv('NGROK_AUTH_TOKEN', '')

if not NGROK_AUTH_TOKEN:
    print('❌ NGROK_AUTH_TOKEN not set.')
    print('   Add it to Colab Secrets (🔑 icon in the left sidebar).')
else:
    # ── 1. Verify the FastAPI server is really up ──────────
    try:
        health = requests.get('http://127.0.0.1:8000/docs', timeout=5)
        print(f'✅ FastAPI health check: HTTP {health.status_code}')
    except Exception as e:
        print(f'⚠️  FastAPI not responding yet: {e}')
        print('   Run Cell 11 first, wait for the 🚀 message, then re-run this cell.')
        raise SystemExit()

    # ── 2. Kill any stale tunnels ──────────────────────────
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    try:
        ngrok.kill()          # close the pyngrok process if it was running
    except Exception:
        pass
    time.sleep(1)             # brief pause so the old process fully exits

    # ── 3. Open a fresh tunnel ─────────────────────────────
    # addr must be a string 'host:port', NOT just the port int.
    tunnel = ngrok.connect(addr='127.0.0.1:8000', proto='http')

    # pyngrok returns a NgrokTunnel object; extract the string URL
    public_url = tunnel.public_url          # e.g. 'https://xxxx.ngrok-free.app'

    # ── 4. Verify the tunnel itself is reachable ───────────
    time.sleep(2)             # give ngrok a moment to establish the tunnel
    try:
        check = requests.get(public_url + '/docs',
                             headers={'ngrok-skip-browser-warning': '1'},
                             timeout=10)
        tunnel_ok = check.status_code == 200
    except Exception:
        tunnel_ok = False

    print('\n' + '=' * 60)
    print(f'🌍 Public URL : {public_url}')
    print(f'📚 Swagger UI : {public_url}/docs')
    if tunnel_ok:
        print('✅ Tunnel verified — the API is reachable from the internet!')
    else:
        print('⚠️  Tunnel opened but remote check timed out.')
        print('   Try visiting the Swagger URL in your browser manually.')
    print('=' * 60)

✅ FastAPI health check: HTTP 200

🌍 Public URL : https://sumption-showmanly-noel.ngrok-free.dev
📚 Swagger UI : https://sumption-showmanly-noel.ngrok-free.dev/docs
✅ Tunnel verified — the API is reachable from the internet!
